# 2 · Embeddings & semantic search

**Core · Live-code · ~25 min**

Goal: turn crew logs into **vectors** (lists of numbers that capture meaning) and search
them by *meaning* instead of by keywords.

**Before you run anything:** make sure Ollama is running and you've pulled `nomic-embed-text`
(from setup). The first embedding call may take a few seconds while the model loads into
memory — that's normal.

Stuck at any step? The finished version is in [`solution/02_embeddings.ipynb`](solution/02_embeddings.ipynb).

## Step 1 — Load the crew logs

The logs live in `crew_logs.jsonl`. The `.jsonl` extension means **JSON Lines**: one JSON
object per line. Each line is a note like `{ "text": "Replaced a clogged CO2 filter...", ... }`.

We also import a helper: `from shared.llm import embed`. This is a tiny function *we* wrote
(in [`../shared/llm.py`](../shared/llm.py)) that sends text to the local embedding model and
returns its vector — so you don't have to deal with any API boilerplate. The two `sys.path`
lines just let this notebook find the `shared` folder at the project root.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))  # let Python find the shared/ folder
import numpy as np
from shared.llm import embed

# Each line of the file is one JSON object; load them all into a list.
logs = [json.loads(line) for line in open("../data/crew_logs.jsonl")]
texts = [log["text"] for log in logs]   # we only need the text of each note

print(f"Loaded {len(texts)} crew logs. Here's the first one:")
texts[0]

## Step 2 — Look at a single embedding

Let's demystify what `embed` actually returns. We'll embed one sentence and peek at the
result. `embed(...)` returns a **list of vectors** (one per input); since we pass one string,
we take element `[0]`.

You'll see the vector is a long list of numbers. Individually they mean nothing to us — but
*collectively* they encode the sentence's meaning, and that's what makes comparison possible.

In [ ]:
sample = embed("oxygen partial pressure is dropping")[0]
print("This one sentence became a vector of", len(sample), "numbers.")
print("The first 8 numbers:", [round(x, 3) for x in sample[:8]])

## Step 3 — Embed every log

Now we embed all the notes at once and stack the vectors into a **NumPy array** — basically a
grid of numbers with one row per log. NumPy makes the math in the next step fast and tidy.

The shape will print as `(number_of_logs, numbers_per_vector)`.

In [ ]:
# TODO: embed all the `texts` and turn the result into a NumPy array named `vectors`.
#   vectors = np.array(embed(texts))
#   print(vectors.shape)   # (num_logs, embedding_dim)

## Step 4 — Measure similarity (cosine similarity)

**Cosine similarity** compares the *direction* of two vectors. The formula is the dot product
of the vectors divided by the product of their lengths — that sounds fancy but it's three
NumPy calls. The result is ~1 for same-meaning texts and ~0 for unrelated ones.

We then build a `search(query)` function that:
1. embeds the query into a vector,
2. compares it against every log's vector with cosine similarity,
3. returns the top-k most similar logs.

This is a complete (tiny) semantic search engine!

In [ ]:
def cosine(a, b):
    # dot product divided by the two vector lengths -> a value roughly in [-1, 1]
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def search(query, k=3):
    # TODO: fill in the three steps below.
    # 1) embed the query:            qv = np.array(embed(query)[0])
    # 2) score every log vector:     scored = [(cosine(qv, v), texts[i]) for i, v in enumerate(vectors)]
    # 3) return the best k, highest score first:
    #                                return sorted(scored, key=lambda s: s[0], reverse=True)[:k]
    ...

## Step 5 — Try it: search by meaning

Now the payoff. Our query talks about oxygen and CO₂. Notice that the top results include
logs about the **CO₂ scrubber** and **filter cartridge** — even though those logs may not
contain the words "oxygen" or "dropping." The model matched on *meaning*, exactly what
keyword search could never do.

In [ ]:
for score, text in search("oxygen is dropping and CO2 rising in Module B"):
    print(f"{score:.3f}  {text}")

### 🌱 Stretch (optional)

- Try other queries: *"the panels are covered in dust"*, *"someone feels unwell"*, *"a suit
  is leaking"*. Each should surface the relevant logs without sharing their exact words.
- Pick two sentences you write yourself and print their cosine similarity. Can you make two
  differently-worded sentences score above 0.7?

## ✅ Recap

You turned text into vectors, measured similarity, and built a search that understands
*meaning*. Hold onto this mental model: **embed → compare → retrieve the closest.** In
Module 4 we'll use the very same three steps on the colony's manuals to give ARIA reliable,
fact-based answers.

# 2 · Embeddings & semantic search

**Core · Live-code · ~25 min**

Turn crew logs into vectors and search them by *meaning*.

> Needs Ollama running with `nomic-embed-text`. Fill in each `# TODO`.
> Solution: [`solution/02_embeddings.ipynb`](solution/02_embeddings.ipynb).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))  # so we can import shared/
import numpy as np
from shared.llm import embed

# Load the crew logs
logs = [json.loads(line) for line in open("../data/crew_logs.jsonl")]
texts = [log["text"] for log in logs]
print(f"Loaded {len(texts)} crew logs")
texts[0]

## What does an embedding look like?

`embed(...)` returns one vector per input string.

In [ ]:
sample = embed("oxygen partial pressure is dropping")[0]
print("vector length:", len(sample))
print("first 8 numbers:", [round(x, 3) for x in sample[:8]])

## Embed all the logs

In [ ]:
# TODO: embed all `texts` and turn the result into a numpy array called `vectors`
# vectors = np.array(embed(texts))
# print(vectors.shape)  # (num_logs, embedding_dim)

## Cosine similarity

Two vectors pointing in a similar direction have cosine similarity near 1.

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def search(query, k=3):
    # TODO: embed the query, score it against every log vector with cosine(),
    #       and return the top-k logs sorted by score.
    # qv = np.array(embed(query)[0])
    # scored = [(cosine(qv, v), texts[i]) for i, v in enumerate(vectors)]
    # return sorted(scored, reverse=True)[:k]
    ...

In [ ]:
for score, text in search("oxygen is dropping and CO2 rising in Module B"):
    print(f"{score:.3f}  {text}")

### 🌱 Stretch

- Try queries like *"the panels are covered in dust"* or *"someone feels unwell"*.
- Notice the search finds relevant logs even when they don't share keywords — that's
  the power we'll use for RAG in Module 4.